# 16. RAG vs. CAG

**Tier:** Building with LLMs
**Estimated time:** 45 minutes
**Prerequisites:** 13, 14, 15
**Priority:** 🟡 Important — caching strategy matters at scale (ties to notebook 32). *If skipped, revisit when:* before a cost/latency pass on a RAG system.
**Source material:** @akshay_pachaar — https://x.com/akshay_pachaar/status/2056714042455343160

## What You'll Learn
- The difference between **Retrieval**-Augmented Generation and **Cache**-Augmented Generation, and when each wins
- How to split your knowledge into "cold" (cacheable) and "hot" (retrievable) layers
- How to measure the latency and token-cost tradeoff between the two, and why Claude's ~92% cache-hit-rate matters

## Why This Matters
Notebook 14 retrieved knowledge; notebook 13 cached it. This notebook puts them head to head. The choice between "stuff the stable knowledge into a cached prompt" and "retrieve the right snippet on demand" is one of the highest-leverage cost/latency decisions in a production LLM system.


## Two ways to give a model knowledge it wasn't trained on

Both RAG and CAG solve the same problem from notebook 14 — the model doesn't know your private data — but they make opposite bets about *when* to pay for that knowledge.

**RAG (Retrieval-Augmented Generation)** keeps knowledge in a vector store (notebook 15) and fetches only the few relevant chunks per query. You pay a retrieval step and only put a small slice of knowledge in the context window each time. Best when the knowledge base is large, changes often, or only a tiny fraction is relevant to any one question.

**CAG (Cache-Augmented Generation)** puts the *entire* relevant knowledge directly into the prompt once, and relies on prompt caching (notebook 13) so that re-sending that big prompt prefix is cheap on every subsequent call. There's no retrieval step and no risk of retrieving the wrong chunk — the model sees everything. Best when the knowledge is small enough to fit in context, stable, and queried repeatedly.

The practical pattern is a **two-layer split**:
- **Cold layer** (stable, cache it): product documentation, policies, API references — rarely changes, so cache it once and re-read it nearly free.
- **Hot layer** (volatile, retrieve it): today's support tickets, live inventory, recent events — changes constantly, so retrieve fresh each time.

Claude's prompt cache reaching a ~92% hit-rate on the cold layer is what makes CAG economical: you write the cache once and read it back dozens of times per dollar.


In [ ]:
import os, time
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

def raw_call(system_blocks, user_text, max_tokens=120, model=TEACH_MODEL):
    """Call Claude with a structured system (list of blocks, so we can mark one cacheable).
    Returns the full message object (for usage stats) or None on failure."""
    if not HAS_ANTHROPIC:
        print("  [skipped: no ANTHROPIC_API_KEY]")
        return None
    try:
        import anthropic
        client = anthropic.Anthropic()
        return client.messages.create(
            model=model, max_tokens=max_tokens,
            system=system_blocks,
            messages=[{"role": "user", "content": user_text}],
        )
    except Exception as e:
        print(f"  [skipped: API call failed — {type(e).__name__}: {str(e)[:150]}]")
        return None


## The knowledge: cold docs + hot tickets

We use a small product-documentation block (the cold/cacheable layer) and a set of recent support tickets (the hot/retrievable layer).


In [ ]:
# COLD layer: stable product documentation (cache this once, re-read cheaply)
PRODUCT_DOCS = """HELIOS X1 PRODUCT MANUAL (stable reference)
- The Helios X1 ships with a 2-year limited warranty covering manufacturing defects.
- Wifi reset: hold the reset button 10 seconds while powered on, then release.
- A solid red LED after reset indicates a firmware mismatch; update via the companion app.
- Operating temperature range: 0C to 40C. Charging above 40C is disabled by design.
- The X1 supports firmware rollback from Settings > Updates > Version History.
""" * 6   # repeated to be comfortably above the cache minimum

# HOT layer: recent support tickets (these change daily — retrieve, don't cache)
SUPPORT_TICKETS = [
    "Ticket 8821: Customer reports X1 LED solid red after a power outage during an update.",
    "Ticket 8822: Customer asks whether the X1 warranty covers water damage.",
    "Ticket 8823: Customer in Dubai says the unit won't charge in a 45C warehouse.",
]
print(f"Cold docs: {len(PRODUCT_DOCS)} chars   Hot tickets: {len(SUPPORT_TICKETS)}")


## Pattern A — CAG: cache the docs, answer from the prompt

We mark the product docs with `cache_control` so the prefix is cached. The first call *creates* the cache; later calls *read* it cheaply.


In [ ]:
def cag_answer(question, max_tokens=120):
    system_blocks = [
        {"type": "text", "text": "You are a Helios support assistant. Answer only from the manual below."},
        {"type": "text", "text": PRODUCT_DOCS, "cache_control": {"type": "ephemeral"}},  # cached prefix
    ]
    return raw_call(system_blocks, question, max_tokens=max_tokens)

# First call creates the cache; second identical-prefix call should read it.
q1 = "How long is the X1 warranty, and does charging work at 45C?"
m1 = cag_answer(q1)
m2 = cag_answer("What does a solid red LED mean after reset?")

for label, m in [("call 1 (cache create)", m1), ("call 2 (cache read)", m2)]:
    if m is not None:
        u = m.usage
        print(f"{label}: input={u.input_tokens} "
              f"cache_create={getattr(u,'cache_creation_input_tokens','n/a')} "
              f"cache_read={getattr(u,'cache_read_input_tokens','n/a')}")
if m1 is not None:
    print("\nAnswer:", m1.content[0].text)


## Pattern B — RAG: retrieve the relevant ticket, then answer

For the hot layer we retrieve instead of caching — the tickets change too often to cache. We reuse the embedding retrieval from notebook 14.


In [ ]:
import sys
sys.path.insert(0, "..")
from ragkit.embeddings import embed, cosine_similarity

ticket_vectors = embed(SUPPORT_TICKETS)

def rag_answer(question, max_tokens=120):
    qv = embed([question])
    sims = cosine_similarity(qv, ticket_vectors)
    best = int(sims.argmax())
    retrieved = SUPPORT_TICKETS[best]
    system_blocks = [
        {"type": "text", "text": "You are a Helios support assistant. Use the retrieved ticket as context."},
    ]
    user = f"<retrieved_ticket>\n{retrieved}\n</retrieved_ticket>\n\nQuestion: {question}"
    return raw_call(system_blocks, user, max_tokens=max_tokens), retrieved

m, retrieved = rag_answer("Is there an open ticket about charging in hot warehouses?")
print("Retrieved:", retrieved)
if m is not None:
    print("Answer:", m.content[0].text)


## Measuring the tradeoff: a simulated repeated workload

CAG's whole economic case is the cache hit rate. We simulate a workload of N queries against the cold layer and tally cache reads vs. creates — and approximate cost, where a cached token is billed at ~10% of a normal input token.


In [ ]:
def simulate_cag_workload(n_queries=20):
    """First query creates the cache; the rest hit it. Returns (hits, total, billed_input_tokens)."""
    questions = [
        "What is the warranty length?", "How do I reset wifi?",
        "What does a red LED mean?", "What's the operating temperature?",
        "Can I roll back firmware?",
    ]
    creates, reads, billed = 0, 0, 0.0
    last = None
    for i in range(n_queries):
        q = questions[i % len(questions)]
        m = cag_answer(q, max_tokens=30)
        if m is None:
            return None
        u = m.usage
        cc = getattr(u, "cache_creation_input_tokens", 0) or 0
        cr = getattr(u, "cache_read_input_tokens", 0) or 0
        creates += 1 if cc > 0 else 0
        reads += 1 if cr > 0 else 0
        # cached reads billed at ~10% of normal input token price
        billed += (u.input_tokens) + cc + 0.1 * cr
        last = u
    return creates, reads, n_queries, billed

result = simulate_cag_workload(n_queries=12)
if result is not None:
    creates, reads, total, billed = result
    hit_rate = reads / total
    print(f"Over {total} queries: cache creates={creates}, cache reads={reads}")
    print(f"Cache hit-rate: {hit_rate:.0%}  (the more repeated the prefix, the closer to ~92%)")
    print(f"Approx billed input units (cached reads at 10%): {billed:,.0f}")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Illustrative cost comparison: same knowledge, served via CAG (cached) vs re-sent uncached.
labels = ["CAG\n(cached prefix)", "Uncached\n(resend docs each call)"]
# 12 calls; cached reads at 10% vs full price each time (illustrative units)
illustrative_cost = [1.0 + 11 * 0.1, 12 * 1.0]
plt.figure(figsize=(6, 4))
plt.bar(labels, illustrative_cost, color=["#55A868", "#C44E52"])
plt.title("Illustrative input-token cost over 12 repeated-prefix calls")
plt.ylabel("Relative cost (cold docs = 1 unit)")
plt.tight_layout(); plt.show()


*Illustrative: once the cold layer is cached, every repeat call pays roughly a tenth for that prefix — which is why CAG only makes sense for stable, frequently-reused knowledge.*


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Break the cache
# Task: In cag_answer, append the current question to the END of the cached PRODUCT_DOCS block
#       (so the cached text changes every call). Re-run simulate_cag_workload — what happens to
#       the hit-rate?
# Hint: The cache only hits when the cached prefix is byte-identical; changing it per call forces
#       a fresh cache_creation every time.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Route by layer
# Task: Write a function answer(question) that decides: if the question is about stable docs
#       (warranty, reset, temperature) use cag_answer; if it's about a specific recent ticket
#       use rag_answer. Test it on one question of each type.
# Hint: A keyword check is enough here; in notebook 18+ an LLM tool-router makes this decision.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): When does CAG lose?
# Task: Grow SUPPORT_TICKETS to ~50 tickets. Argue (in a comment) and demonstrate why caching ALL
#       of them in the prompt becomes a worse choice than retrieving — consider both token cost
#       per call and staleness.
# Hint: A cached prefix is only cheap to RE-READ; you still pay full price to CREATE it, and you
#       pay it again every time the ticket list changes. Retrieval sends only the 1 relevant ticket.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
def cag_answer_broken(question, max_tokens=30):
    system_blocks = [
        {"type": "text", "text": "You are a Helios support assistant."},
        {"type": "text", "text": PRODUCT_DOCS + question,  # prefix changes every call!
         "cache_control": {"type": "ephemeral"}},
    ]
    return raw_call(system_blocks, question, max_tokens=max_tokens)
m = cag_answer_broken("test")
if m: print("cache_read:", getattr(m.usage, "cache_read_input_tokens", 0))  # stays ~0: no hits

# Exercise 2
def answer(question):
    stable_keywords = ("warranty", "reset", "temperature", "led", "firmware", "rollback")
    if any(k in question.lower() for k in stable_keywords):
        m = cag_answer(question); return m.content[0].text if m else None
    m, _ = rag_answer(question); return m.content[0].text if m else None
print(answer("How long is the warranty?"))
print(answer("Any ticket about a hot warehouse?"))

# Exercise 3
# With ~50 tickets, caching the whole list means: (a) a large cache_creation cost paid again
# every time ANY ticket is added/edited (the prefix changes -> cache invalidated), and (b) the
# model wades through 49 irrelevant tickets per query. Retrieval sends only the 1 best-matching
# ticket, so per-call tokens stay flat as the ticket list grows. CAG wins on STABLE knowledge;
# RAG wins on VOLATILE, large knowledge.
many_tickets = SUPPORT_TICKETS + [f"Ticket {9000+i}: misc issue {i}" for i in range(47)]
print(f"Caching {len(many_tickets)} volatile tickets would invalidate on every edit — retrieve instead.")
```
</details>


## Key Takeaways
- RAG fetches a small relevant slice per query; CAG puts the whole knowledge in a cached prompt prefix and re-reads it cheaply.
- Split knowledge into a **cold** layer (stable → cache via CAG) and a **hot** layer (volatile → retrieve via RAG).
- CAG's economics depend on the cache hit-rate; a byte-identical prefix is what makes Claude's ~92% hit-rate — and the cheap re-reads — possible.
- CAG wins for small, stable, frequently-queried knowledge; RAG wins for large or fast-changing knowledge where only a fraction is relevant per query.
- The two compose: cache the manual, retrieve today's tickets, in the same request.

## What's Next
Notebook **17 — Chain of Thought** shifts from *what knowledge you give the model* to *how you make it reason* over that knowledge — getting better answers by having the model think in steps before it commits.
